# Project 9 – Data Engineering, Risk Analysis and Dimensionality Reduction

This notebook implements a data engineering and machine learning workflow on real-world loan risk data.  
It also includes dimensionality reduction experiments on MNIST using PCA, LDA and UMAP.

The project is divided into five parts:

1. Data profiling and exploration
2. Data preprocessing and visualization
3. Classification for loan candidate selection
4. Feature importance and feature selection
5. Dimensionality reduction on MNIST

## 0. Imports and Dataset Upload

The loan dataset is uploaded directly in Google Colab. The file may be named `bankloan.csv`, `Case_Data.csv`, or any other CSV filename from the Kaggle archive.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

uploaded = files.upload()
csv_file = list(uploaded.keys())[0]

bank = pd.read_csv(csv_file)
bank.columns = bank.columns.str.strip()

print("Dataset shape:", bank.shape)
print("Columns:")
print(bank.columns.tolist())
display(bank.head())

## 1. Data Profiling and Description

First, the dataset is explored using standard pandas methods.  
Since the dataset contains many variables, a profiling report can also be generated using `ydata-profiling`.

In [ ]:
display(bank.describe(include="all").T)

missing_table = pd.DataFrame({
    "missing_values": bank.isnull().sum(),
    "missing_percentage": bank.isnull().mean() * 100,
    "dtype": bank.dtypes
}).sort_values("missing_percentage", ascending=False)

display(missing_table)

### Optional: ydata-profiling report

This cell creates an HTML profiling report. If Colab is slow, run it once, download the HTML file, and open it in a browser.

In [ ]:
# Optional profiling report. Uncomment if needed.
# !pip install -q ydata-profiling
# from ydata_profiling import ProfileReport
# profile = ProfileReport(bank, title="Bank Loan Profiling Report", explorative=True, minimal=True)
# profile.to_file("bankloan_profile.html")
# files.download("bankloan_profile.html")

## 2. Data Preprocessing and Visualization

In this section we compute basic loan amount statistics, remove variables that are not useful for model training, create the target variable, and perform preprocessing.

The target is defined as follows:

- 1: the applicant belongs to category `A`, `B1`, or `B2`
- 0: otherwise

This represents applicants that the bank would prioritize for loan approval.

In [ ]:
loan_stats = bank["loan_amnt"].agg(["mean", "min", "max"])
print("Loan amount statistics:")
display(loan_stats)

plt.figure(figsize=(8, 5))
plt.hist(bank["loan_amnt"].dropna(), bins=40)
plt.title("Distribution of Requested Loan Amount")
plt.xlabel("Loan Amount")
plt.ylabel("Frequency")
plt.show()

In [ ]:
bank_work = bank.copy()

bank_work["target"] = (
    (bank_work["grade"].astype(str).str.upper() == "A") |
    (bank_work["sub_grade"].astype(str).str.upper().isin(["B1", "B2"]))
).astype(int)

target_counts = bank_work["target"].value_counts()
target_percentages = bank_work["target"].value_counts(normalize=True) * 100

target_table = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

display(target_table)

plt.figure(figsize=(5, 4))
bank_work["target"].value_counts().sort_index().plot(kind="bar")
plt.title("Target Distribution")
plt.xlabel("Target")
plt.ylabel("Count")
plt.xticks(ticks=[0, 1], labels=["Not Selected", "Selected"], rotation=0)
plt.show()

### Target interpretation

The target variable is imbalanced because only a subset of applicants belongs to the highest-priority credit categories. This is expected in a risk analysis application, since banks usually approve or prioritize only the strongest applicants.

In [ ]:
loan_bins = pd.interval_range(
    start=bank_work["loan_amnt"].min(),
    end=bank_work["loan_amnt"].max() + 5000,
    freq=5000,
    closed="left"
)

bank_work["loan_amount_range"] = pd.cut(bank_work["loan_amnt"], bins=loan_bins)

range_table = bank_work.groupby("loan_amount_range").agg(
    total_applicants=("target", "count"),
    selected_applicants=("target", "sum"),
    selected_percentage=("target", "mean")
).reset_index()

range_table["selected_percentage"] = range_table["selected_percentage"] * 100
accepted_ranges = range_table[(range_table["selected_percentage"] >= 15) & (range_table["total_applicants"] >= 30)]

display(range_table)
print("Ranges with at least 15% selected applicants:")
display(accepted_ranges)

### Variables removed before modeling

Some variables are removed because they are identifiers, highly sparse, text-heavy, duplicated, or not available at the time of application.  
The variables `grade`, `sub_grade`, and `int_rate` are also removed because they directly reveal or strongly encode the target decision.

In [ ]:
columns_to_drop = [
    "Row ID", "id", "member_id", "Unnamed: 50",
    "grade", "sub_grade", "int_rate",
    "emp_title", "title",
    "issue_d", "earliest_cr_line", "last_pymnt_d", "next_pymnt_d", "last_credit_pull_d",
    "loan_status", "out_prncp", "total_pymnt", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee", "last_pymnt_amnt",
    "36months", "60months", "loan_amount_range"
]

existing_drop_cols = [col for col in columns_to_drop if col in bank_work.columns]
model_df = bank_work.drop(columns=existing_drop_cols)

missing_ratio = model_df.isnull().mean()
high_missing_cols = missing_ratio[missing_ratio > 0.60].index.tolist()
model_df = model_df.drop(columns=high_missing_cols)

print("Dropped high-missing columns:", high_missing_cols)
print("Model dataframe shape:", model_df.shape)
display(model_df.head())

## 3. Classification

A Logistic Regression classifier is used as a baseline model.  
It is appropriate because it is fast, interpretable, and works well with normalized numerical inputs and one-hot encoded categorical variables.

The preprocessing pipeline applies:

- median imputation and Standard Scaling to numerical features
- most-frequent imputation and One-Hot Encoding to categorical features

The split is stratified to preserve the class distribution in both train and test sets.

In [ ]:
X = model_df.drop(columns=["target"])
y = model_df["target"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=0,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
clf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1))
])

param_grid = {
    "classifier__C": [0.1, 1.0, 10.0]
}

grid = GridSearchCV(
    clf_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Best parameters:", grid.best_params_)
print("Best validation F1:", grid.best_score_)

y_pred = best_model.predict(X_test)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred)
}

display(pd.DataFrame([metrics]))
print(classification_report(y_test, y_pred))
display(pd.DataFrame(confusion_matrix(y_test, y_pred), index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"]))

### Metric interpretation

For this application, precision and recall are more informative than accuracy because the target is imbalanced.  
Precision shows how many selected applicants are actually strong candidates, while recall shows how many strong candidates the model manages to identify.  
F1-score is useful because it combines precision and recall into one metric.

## 4. Feature Importance and Selection

A Random Forest classifier is trained in order to estimate feature importance.  
Random Forests are useful here because they can model non-linear relationships and provide an importance score for each feature.

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        random_state=0,
        class_weight="balanced",
        n_jobs=-1,
        max_depth=12
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

rf_metrics = {
    "Accuracy": accuracy_score(y_test, rf_pred),
    "F1": f1_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred),
    "Recall": recall_score(y_test, rf_pred)
}

display(pd.DataFrame([rf_metrics]))

In [ ]:
fitted_preprocessor = rf_pipeline.named_steps["preprocessor"]
rf_model = rf_pipeline.named_steps["classifier"]

feature_names = fitted_preprocessor.get_feature_names_out()
importances = rf_model.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

top15 = importance_df.head(15)
display(top15)

plt.figure(figsize=(10, 6))
plt.barh(top15["feature"][::-1], top15["importance"][::-1])
plt.title("Top 15 Feature Importances - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

In [ ]:
X_train_transformed = fitted_preprocessor.transform(X_train)
X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=feature_names)

top15_features = top15["feature"].tolist()
top15_corr = X_train_transformed_df[top15_features].corr()

display(top15_corr)

plt.figure(figsize=(10, 8))
plt.imshow(top15_corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(top15_features)), top15_features, rotation=90)
plt.yticks(range(len(top15_features)), top15_features)
plt.title("Correlation Matrix of Top 15 Features")
plt.show()

In [ ]:
selected_features = []
for feature in top15_features:
    if len(selected_features) == 0:
        selected_features.append(feature)
    else:
        correlations = top15_corr.loc[feature, selected_features].abs()
        if (correlations < 0.75).all():
            selected_features.append(feature)

print("Selected low-correlation features:")
print(selected_features)

selected_idx = [list(feature_names).index(f) for f in selected_features]
X_train_selected = X_train_transformed[:, selected_idx]
X_test_selected = fitted_preprocessor.transform(X_test)[:, selected_idx]

rf_selected = RandomForestClassifier(
    n_estimators=150,
    random_state=0,
    class_weight="balanced",
    n_jobs=-1,
    max_depth=12
)

rf_selected.fit(X_train_selected, y_train)
selected_pred = rf_selected.predict(X_test_selected)

selected_metrics = {
    "Accuracy": accuracy_score(y_test, selected_pred),
    "F1": f1_score(y_test, selected_pred),
    "Precision": precision_score(y_test, selected_pred),
    "Recall": recall_score(y_test, selected_pred)
}

display(pd.DataFrame([selected_metrics]))
print(classification_report(y_test, selected_pred))

## 5. Dimensionality Reduction on MNIST

In this part, MNIST handwritten digit images are used to compare dimensionality reduction methods.

The original images have 28 × 28 = 784 pixel features.  
Standard Scaling is applied before PCA, LDA and UMAP.

In [ ]:
from tensorflow.keras.datasets import mnist
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.neighbors import KNeighborsClassifier

(x_train_img, y_train_mnist), (x_test_img, y_test_mnist) = mnist.load_data()

x_train_flat = x_train_img.reshape(x_train_img.shape[0], -1)
x_test_flat = x_test_img.reshape(x_test_img.shape[0], -1)

print("Original train shape:", x_train_img.shape)
print("Flattened train shape:", x_train_flat.shape)

### Practical Colab setting

KNN on the full MNIST dataset can be slow because KNN stores all training examples and compares new samples against them.  
For faster experimentation, the following cell uses a subset. Set `USE_FULL_MNIST = True` for the full dataset.

In [ ]:
USE_FULL_MNIST = False
TRAIN_LIMIT = 20000
TEST_LIMIT = 5000

if USE_FULL_MNIST:
    X_train_mnist = x_train_flat
    y_train_small = y_train_mnist
    X_test_mnist = x_test_flat
    y_test_small = y_test_mnist
else:
    X_train_mnist = x_train_flat[:TRAIN_LIMIT]
    y_train_small = y_train_mnist[:TRAIN_LIMIT]
    X_test_mnist = x_test_flat[:TEST_LIMIT]
    y_test_small = y_test_mnist[:TEST_LIMIT]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_mnist)
X_test_scaled = scaler.transform(X_test_mnist)

print("Training samples used:", X_train_scaled.shape[0])
print("Test samples used:", X_test_scaled.shape[0])

### PCA to 300 components

PCA is fitted only on the training set. The test set is transformed using the already fitted PCA model.  
If PCA were fitted separately on the train and test sets, the two datasets would be projected into different feature spaces.  
If PCA were fitted on the combined train and test data, information from the test set would leak into training.

In [ ]:
pca = PCA(n_components=300, random_state=0)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

ks = [5, 15, 51, 101]
pca_knn_results = []

for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train_pca, y_train_small)
    train_acc = knn.score(X_train_pca, y_train_small)
    test_acc = knn.score(X_test_pca, y_test_small)
    pca_knn_results.append({"k": k, "train_accuracy": train_acc, "test_accuracy": test_acc})

pca_knn_df = pd.DataFrame(pca_knn_results)
display(pca_knn_df)

plt.figure(figsize=(7, 5))
plt.plot(pca_knn_df["k"], pca_knn_df["train_accuracy"], marker="o", label="Train")
plt.plot(pca_knn_df["k"], pca_knn_df["test_accuracy"], marker="o", label="Test")
plt.title("KNN Accuracy after PCA")
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### LDA to 2 components

LDA is supervised and uses the class labels to find directions that separate the digits as much as possible.  
The 2D scatter plot shows how well the method separates the digit classes in two dimensions.

In [ ]:
lda = LDA(n_components=2)
X_train_lda = lda.fit_transform(X_train_scaled, y_train_small)
X_test_lda = lda.transform(X_test_scaled)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(X_train_lda[:, 0], X_train_lda[:, 1], c=y_train_small, s=5, alpha=0.6, cmap="tab10")
plt.colorbar(scatter, ticks=range(10))
plt.title("MNIST Visualization using LDA")
plt.xlabel("LDA Component 1")
plt.ylabel("LDA Component 2")
plt.show()

### UMAP to 2 components

UMAP is a non-linear dimensionality reduction method. It often preserves local neighborhood structure better than linear methods and can create clearer visual clusters.

In [ ]:
!pip install -q umap-learn

import umap.umap_ as umap

UMAP_LIMIT = min(10000, X_train_scaled.shape[0])

umap_model = umap.UMAP(
    n_components=2,
    random_state=0,
    n_neighbors=15,
    min_dist=0.1
)

X_train_umap = umap_model.fit_transform(X_train_scaled[:UMAP_LIMIT])
y_umap = y_train_small[:UMAP_LIMIT]

plt.figure(figsize=(9, 7))
scatter = plt.scatter(X_train_umap[:, 0], X_train_umap[:, 1], c=y_umap, s=5, alpha=0.6, cmap="tab10")
plt.colorbar(scatter, ticks=range(10))
plt.title("MNIST Visualization using UMAP")
plt.xlabel("UMAP Component 1")
plt.ylabel("UMAP Component 2")
plt.show()

## Final Notes

The loan risk analysis part demonstrates a complete data engineering workflow: profiling, preprocessing, target construction, classification, feature importance and feature selection.

The MNIST part shows how dimensionality reduction can reduce the feature space and help visualize high-dimensional image data. PCA is useful for compression, LDA uses labels to improve class separation, and UMAP can reveal non-linear cluster structures.